# Agent evaluation harness — runnable example (Phase V)

> Notebook generated from [`examples/agent_eval.py`](https://github.com/prismal-ai/prismal/blob/main/examples/agent_eval.py).

| Field | Value |
|------|-------|
| **API key** | 🔑 Not required — runs offline with injected fakes |
| **Source** | `examples/agent_eval.py` |

Runs a tiny eval-set through an `EvalRunner` with *injected fakes* (a scripted
graph + a no-op runtime), so it executes offline with no API keys. Swap the
injected factories for the defaults (`EvalRunner()`) to run against the real
compiled graph with `build_test_runtime` fakes.

```bash
uv run python examples/agent_eval.py
```


In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports

In [ ]:
from __future__ import annotations

import asyncio
from typing import TYPE_CHECKING, Any

from langchain_core.messages import AIMessage

if TYPE_CHECKING:
    from collections.abc import AsyncIterator

from prismal.eval.regression import compare
from prismal.eval.report import to_json, to_markdown
from prismal.eval.runner import EvalRunner
from prismal.eval.types import Assertion, AssertionType, EvalCase, EvalSet

## Injected fakes — scripted graph and no-op runtime

In [ ]:
class _ScriptedGraph:
    """A graph double whose astream replays a fixed answer for any input."""

    def __init__(self, answer: str) -> None:
        self._answer = answer

    async def astream(
        self, _input: Any, _config: Any = None, *, stream_mode: str = "updates"
    ) -> AsyncIterator[dict[str, Any]]:
        yield {"supervisor": {"messages": [AIMessage(content=self._answer)]}}

In [ ]:
class _NoopRuntime:
    tool_provider = None

    async def aclose(self) -> None:
        return None

## Building the runner

In [ ]:
def _runner(answer: str) -> EvalRunner:
    async def graph_factory(**_k: Any) -> Any:
        return _ScriptedGraph(answer)

    def runtime_factory(**_k: Any) -> Any:
        return _NoopRuntime()

    return EvalRunner(graph_factory=graph_factory, runtime_factory=runtime_factory)

## The demo

In [ ]:
async def main() -> None:
    eval_set = EvalSet(
        suite="example",
        cases=[
            EvalCase(
                id="greeting",
                input="Say hi.",
                assertions=[Assertion(type=AssertionType.EXACT, expected="hi there")],
            ),
            EvalCase(
                id="tool-bounds",
                input="Answer directly.",
                assertions=[Assertion(type=AssertionType.TOOL_USAGE, max_steps=3)],
            ),
        ],
    )

    card = await _runner("hi there").run_set(eval_set)
    print(to_markdown(card))
    print("--- JSON ---")
    print(to_json(card))

    # Regression gate: compare against a (here, identical) baseline.
    result = compare(card, card)
    print(f"\nregression gate passed: {result.passed}")

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (runs offline — no API key needed).
# await main()